# temperature와 max_tokens

`1_1_openai.ipynb`에서는 모델과 메시지만 지정해서 API를 호출했습니다.
이번에는 답변의 성격을 바꾸는 두 가지 파라미터를 직접 비교해 봅니다.

| 파라미터 | 조절하는 것 | 이 노트북에서 볼 것 |
|---|---|---|
| `temperature` | 답변의 **무작위성** | 0이면 매번 같은 답, 1이면 매번 다른 답 |
| `max_tokens` | 답변의 **최대 길이** (출력 토큰 수) | 값이 작으면 문장 중간에서 잘림 |

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [3]:
openai = OpenAI()

# temperature를 지원하는 모델을 사용합니다. (gpt-5 계열은 지원하지 않음 - 아래 1-3 참고)
MODEL = "gpt-4.1-mini"

## 1. temperature: 0 vs 1

LLM은 답변을 한 토큰씩 만듭니다. 매번 "다음에 올 토큰"의 후보와 확률을 계산한 뒤 그중 하나를 고릅니다.
`temperature`는 **이 고르는 과정이 얼마나 무작위적인지**를 정합니다.

- **temperature = 0**: 항상 가장 확률이 높은 토큰을 고릅니다 → **매번 (거의) 같은 답**
- **temperature = 1**: 확률에 비례해서 무작위로 고릅니다 → **매번 다른 답**

차이를 극단적으로 보기 위해 **정답이 없는 질문**(카페 이름 짓기)을 같은 설정으로 5번 물어봅니다.
`n=5`를 주면 한 번의 요청으로 답변 5개를 받을 수 있습니다.

In [4]:
messages = [
    {"role": "user", "content": "새로 여는 동네 카페 이름을 하나만 지어 주세요. 이름만 답하세요."}
]

In [5]:
def compare_temperature(messages, temperature, n=5):
    """같은 질문에 대한 답변 n개를 받아서 출력하고, 서로 다른 답변이 몇 개인지 셉니다."""
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        n=n,
    )
    answers = [choice.message.content for choice in response.choices]

    print(f"===== temperature = {temperature} =====")
    for i, answer in enumerate(answers, 1):
        print(f"{i}. {answer}")
    print(f"→ 서로 다른 답변: {len(set(answers))}개 / {n}개")

In [6]:
compare_temperature(messages, temperature=0)

===== temperature = 0 =====
1. 모닝브루
2. 모닝브루
3. 모닝브루
4. 모닝브루
5. 모닝브루
→ 서로 다른 답변: 1개 / 5개


In [6]:
compare_temperature(messages, temperature=1)

===== temperature = 1 =====
1. 달빛마루
2. 마실커피
3. 모닝듀 카페
4. 따뜻한 오후
5. 모모트리 카페
→ 서로 다른 답변: 5개 / 5개


### 결과

- **temperature = 0**: 5번 모두 같은 이름이 나옵니다. 모델이 매번 가장 확률이 높은 선택만 하기 때문입니다.
- **temperature = 1**: 5번 모두 다른 이름이 나옵니다. 같은 질문, 같은 모델인데도 결과가 달라집니다.

> temperature = 0이 **100% 같은 답을 보장하지는 않습니다.** 서버 내부 계산 방식 때문에 긴 답변에서는 가끔 조금씩 달라질 수 있습니다.
> "항상 같은 답"이 아니라 "가장 일관된 답"이라고 이해하면 됩니다.

### 1-2. 긴 문장에서도 비교하기

이름처럼 짧은 답이 아니라 문장을 만들 때도 차이가 드러납니다.

In [7]:
story_messages = [
    {"role": "user", "content": "'비 오는 월요일 아침,'으로 시작하는 짧은 이야기의 첫 문장을 하나만 써 주세요."}
]

In [8]:
compare_temperature(story_messages, temperature=0, n=3)

===== temperature = 0 =====
1. 비 오는 월요일 아침, 창밖을 바라보던 나는 잊고 있던 추억 하나가 조용히 떠올랐다.
2. 비 오는 월요일 아침, 창밖을 바라보던 나는 잃어버린 기억 한 조각이 빗방울 사이로 스며드는 것을 느꼈다.
3. 비 오는 월요일 아침, 창밖을 바라보던 나는 잃어버린 기억 한 조각이 빗방울 사이로 스며드는 것을 느꼈다.
→ 서로 다른 답변: 2개 / 3개


In [9]:
compare_temperature(story_messages, temperature=1, n=3)

===== temperature = 1 =====
1. 비 오는 월요일 아침, 창밖을 바라보던 나는 잊고 있던 기억 하나가 천천히 떠올랐다.
2. 비 오는 월요일 아침, 창문에 맺힌 빗방울 소리만이 고요한 방안을 채우고 있었다.
3. 비 오는 월요일 아침, 창밖으로 떨어지는 빗방울 소리에 눈을 뜬 나는 차가운 커피 한 모금을 마셨다.
→ 서로 다른 답변: 3개 / 3개


### 언제 어떤 값을 쓰나요?

| temperature | 특징 | 어울리는 작업 |
|---|---|---|
| **0** (낮게) | 일관되고 예측 가능 | 분류, 정보 추출, 요약, 코드 작성, 사실 질문 |
| **1** (높게) | 다양하고 창의적 | 아이디어 내기, 이름 짓기, 글쓰기, 브레인스토밍 |

OpenAI API에서 temperature는 **0 ~ 2** 사이 값을 쓸 수 있고, 지정하지 않으면 기본값은 1입니다.
2에 가까워질수록 점점 엉뚱한 답이 나옵니다.

### 1-3. 주의: 모든 모델이 temperature를 지원하지는 않습니다

`1_1_openai.ipynb`에서 사용한 `gpt-5-nano` 같은 **추론(reasoning) 모델**은 temperature를 바꿀 수 없습니다.
기본값(1)만 허용하기 때문에, 0을 넣으면 오류가 납니다.

In [10]:
try:
    openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,
        temperature=0,
    )
except Exception as e:
    print(f"[오류] {e}")

[오류] Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0 with this model. Only the default (1) value is supported.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': 'unsupported_value'}}


## 2. max_tokens: 출력 길이 제한

`max_tokens`는 모델이 **생성할 수 있는 토큰 수의 상한**입니다.
답변이 끝나기 전에 상한에 도달하면, 모델은 **문장 중간이라도 그 자리에서 멈춥니다.**

같은 질문에 `max_tokens`를 1, 5, 20, 제한 없음으로 바꿔 가며 보내 봅니다.
temperature는 0으로 고정해서, 달라지는 것이 **길이뿐**이 되도록 합니다.

In [11]:
explain_messages = [
    {"role": "user", "content": "인공지능에 대해 세 문장으로 설명해 주세요."}
]

In [12]:
for max_tokens in [1, 5, 20, None]:
    options = {} if max_tokens is None else {"max_tokens": max_tokens}

    response = openai.chat.completions.create(
        model=MODEL,
        messages=explain_messages,
        temperature=0,
        **options,
    )
    choice = response.choices[0]

    print(f"===== max_tokens = {max_tokens if max_tokens else '제한 없음'} =====")
    print(f"finish_reason : {choice.finish_reason}")
    print(f"출력 토큰 수  : {response.usage.completion_tokens}")
    print(f"답변          : {choice.message.content}")
    print()

===== max_tokens = 1 =====
finish_reason : length
출력 토큰 수  : 1
답변          : 인

===== max_tokens = 5 =====
finish_reason : length
출력 토큰 수  : 5
답변          : 인공지능은 컴

===== max_tokens = 20 =====
finish_reason : length
출력 토큰 수  : 20
답변          : 인공지능은 컴퓨터가 인간처럼 학습하고 문제를 해결할 수 있도록 만드는 기술

===== max_tokens = 제한 없음 =====
finish_reason : stop
출력 토큰 수  : 64
답변          : 인공지능은 컴퓨터가 인간처럼 학습하고 문제를 해결할 수 있도록 만드는 기술입니다. 이를 통해 음성 인식, 이미지 분석, 자연어 처리 등 다양한 분야에서 활용됩니다. 인공지능은 빅데이터와 알고리즘을 기반으로 점점 더 발전하고 있습니다.



### 결과

| max_tokens | finish_reason | 결과 |
|---|---|---|
| 1 | `length` | 한 글자만 나오고 끝남 |
| 5 | `length` | 단어 중간에서 잘림 |
| 20 | `length` | 거의 완성된 문장처럼 보이지만 **잘린 것** |
| 제한 없음 | `stop` | 모델이 스스로 답변을 끝냄 |

**`finish_reason`으로 잘렸는지 확인할 수 있습니다.**

- `stop`: 모델이 답변을 스스로 마쳤습니다.
- `length`: `max_tokens`에 걸려서 **강제로 잘렸습니다.**

max_tokens = 20의 결과처럼, 눈으로 보면 멀쩡해 보여도 실제로는 잘린 답변일 수 있습니다.
그래서 프로그램에서는 답변 내용이 아니라 `finish_reason`을 확인해야 합니다.

> 한국어는 영어보다 같은 내용에 더 많은 토큰을 씁니다 (`1_5_tokenizer.ipynb` 참고).
> 그래서 같은 max_tokens라도 한국어 답변이 더 빨리 잘립니다.

### 2-2. 프로그램에서 잘림 확인하기

In [13]:
response = openai.chat.completions.create(
    model=MODEL,
    messages=explain_messages,
    max_tokens=20,
)

if response.choices[0].finish_reason == "length":
    print("⚠ 답변이 max_tokens에 걸려 잘렸습니다. max_tokens를 늘려 주세요.")

print(response.choices[0].message.content)

⚠ 답변이 max_tokens에 걸려 잘렸습니다. max_tokens를 늘려 주세요.
인공지능(AI)은 인간의 지능을 모방하여 학습하고 문제를 해결


## 정리

| | temperature | max_tokens |
|---|---|---|
| 조절하는 것 | 답변의 무작위성 | 답변의 최대 길이 |
| 작은 값 | 매번 같은 답 | 문장 중간에서 잘림 |
| 큰 값 | 매번 다른 답 | 모델이 스스로 끝낼 때까지 생성 |
| 확인 방법 | 같은 질문을 여러 번 보내 비교 | `finish_reason`이 `length`인지 확인 |

**기억하기**

- temperature는 "얼마나 모험적으로 단어를 고를지", max_tokens는 "얼마나 길게 말할 수 있는지"입니다.
- max_tokens는 답변을 **짧게 만드는** 기능이 아니라 **자르는** 기능입니다. 짧은 답이 필요하면 프롬프트에 "한 문장으로" 처럼 요청하세요.
- max_tokens는 출력 토큰 요금이 예상보다 커지는 것을 막는 **안전장치**로 쓰기 좋습니다.
- 최신 모델(gpt-5 계열 등)은 `max_tokens` 대신 `max_completion_tokens`를 사용합니다.